<a href="https://colab.research.google.com/github/LaibaSabir1/flyrank-ml-internship-laiba_sabir/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaSabir1/flyrank-ml-internship-laiba_sabir/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/LaibaSabir1/flyrank-ml-internship-laiba_sabir"
REPO_DIR = "flyrank-ml-internship-laiba_sabir"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-laiba_sabir
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [2]:
from IPython.display import Markdown, display

display(Markdown("""
This is a ranking/scoring task, not classification. I am not predicting a category.
I am producing a continuous "review priority" score per page so the team knows which
pages to look at first. Ranking fits because the real decision is not "yes/no refresh,"
it is "what order do we work through 30k pages in, given limited review time."
"""))


This is a ranking/scoring task, not classification. I am not predicting a category.
I am producing a continuous "review priority" score per page so the team knows which
pages to look at first. Ranking fits because the real decision is not "yes/no refresh,"
it is "what order do we work through 30k pages in, given limited review time."


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [3]:
display(Markdown("""
**Target/proxy:** `is_declining_label`, defined as `trend_direction == "down"`.

**Where it comes from:** it's a **rule**, not a directly-observed outcome. The `trend_direction` column itself is computed from `trend_pct`, which compares the last 30 days of impressions to the previous 30 days (down = more than a 20% drop).
So this is a **proxy label** for "this page needs attention" — I can't directly measure "needs review," so I'm substituting a measurable stand-in: recent search decline.

**Why it's an honest starting proxy, but not perfect:** it only looks at the current window, not a future outcome.
A stronger version (for later weeks) would define the label as "declined over the *next* 30 days," predicted from *only* the prior 90 days of features — that avoids leakage and tests real predictive power instead of just describing the present.

**What I will never do:** use `trend_direction` or `trend_pct` themselves as model features, since the label is derived directly from them — that would be leakage
(the model would just be reading the answer off a mirror).
"""))


**Target/proxy:** `is_declining_label`, defined as `trend_direction == "down"`.

**Where it comes from:** it's a **rule**, not a directly-observed outcome. The `trend_direction` column itself is computed from `trend_pct`, which compares the last 30 days of impressions to the previous 30 days (down = more than a 20% drop).
So this is a **proxy label** for "this page needs attention" — I can't directly measure "needs review," so I'm substituting a measurable stand-in: recent search decline.

**Why it's an honest starting proxy, but not perfect:** it only looks at the current window, not a future outcome.
A stronger version (for later weeks) would define the label as "declined over the *next* 30 days," predicted from *only* the prior 90 days of features — that avoids leakage and tests real predictive power instead of just describing the present.

**What I will never do:** use `trend_direction` or `trend_pct` themselves as model features, since the label is derived directly from them — that would be leakage
(the model would just be reading the answer off a mirror).


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [4]:
display(Markdown("""
**Success metric:** Precision@50 — of the top 50 pages my ranking flags first, what fraction are actually declining (by the label above)?

**Why this metric, not accuracy:** the real decision isn't "classify every page correctly," it's "which ~50 pages does a reviewer check this week, given limited time?"
Precision@50 matches that decision directly — it tells me how many of the reviewer's actual first 50 stops were worth the visit.

**What "good" looks like:** the transparent hand-written baseline rule gets **Precision@50 = 0.24** (~12 of the top 50 right). A trained model on this same
data reaches **~0.74** (~37 of the top 50 right) — so "good" for my lane means beating 0.24 by a meaningful margin, not just matching it.

**Also worth tracking:** Recall and Average Precision, as secondary metrics — Precision@50 alone doesn't tell me how many *total* declining pages I'm missing.
"""))


**Success metric:** Precision@50 — of the top 50 pages my ranking flags first, what fraction are actually declining (by the label above)?

**Why this metric, not accuracy:** the real decision isn't "classify every page correctly," it's "which ~50 pages does a reviewer check this week, given limited time?" 
Precision@50 matches that decision directly — it tells me how many of the reviewer's actual first 50 stops were worth the visit.

**What "good" looks like:** the transparent hand-written baseline rule gets **Precision@50 = 0.24** (~12 of the top 50 right). A trained model on this same
data reaches **~0.74** (~37 of the top 50 right) — so "good" for my lane means beating 0.24 by a meaningful margin, not just matching it.

**Also worth tracking:** Recall and Average Precision, as secondary metrics — Precision@50 alone doesn't tell me how many *total* declining pages I'm missing.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("One row =  one pseudonymized content page (a single piece of published content).")

df[[
    "content_id",
    "client_id",
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "trend_direction",
]].head(5)

Shape: (30000, 44)
One row =  one pseudonymized content page (a single piece of published content).


,content_id,client_id,impressions_90d,avg_position,ctr,days_since_last_update,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,20,down
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,25,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,20,down
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,22,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,14,down


In [6]:
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("Target column preview:")
df[["content_id", "trend_direction", "is_declining_label"]].head(10)

Target column preview:


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


In [7]:
display(Markdown(f"""
**What this confirms:** each row is one content page (`content_id`), belonging to one client (`client_id`). There are **{len(df):,} pages** across **{df['client_id'].nunique()} clients** in this starter slice.
The target column `is_declining_label` is just a 0/1 flag built from `trend_direction`, and it sits right next to the identifying columns — it is NOT one of the feature columns I'd feed a model.
"""))


**What this confirms:** each row is one content page (`content_id`), belonging to one client (`client_id`). There are **30,000 pages** across **32 clients** in this starter slice. 
The target column `is_declining_label` is just a 0/1 flag built from `trend_direction`, and it sits right next to the identifying columns — it is NOT one of the feature columns I'd feed a model.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [8]:
display(Markdown("""
**Why a simple if-statement isn't enough:** a page's "worth reviewing" status depends on many signals at once — visibility (impressions), freshness (days since update),
position, CTR, engagement, content age, word count — and these interact in ways that are hard to write as clean thresholds.
For example, "stale AND visible" is a reasonable rule, but it ignores position, CTR, and engagement entirely, and it can't weigh a small drop in one signal against a big improvement in another.

**The evidence:** on this exact dataset, the transparent hand-written rule (`stale AND visible`) scores **Precision@50 = 0.24**.
 A trained model using the same underlying signals scores **~0.74** — roughly **3x** better. That gap shows there is real, tangled signal in the data that a fixed rule can't capture, but a model can learn.

**Where a rule stays useful:** the rule is still valuable as a transparent baseline and a sanity check — if the model's picks look wildly different from the rule's picks, that's worth investigating. But as the *primary* ranking method, the rule leaves a lot of value on the table.
"""))


**Why a simple if-statement isn't enough:** a page's "worth reviewing" status depends on many signals at once — visibility (impressions), freshness (days since update),
position, CTR, engagement, content age, word count — and these interact in ways that are hard to write as clean thresholds. 
For example, "stale AND visible" is a reasonable rule, but it ignores position, CTR, and engagement entirely, and it can't weigh a small drop in one signal against a big improvement in another.

**The evidence:** on this exact dataset, the transparent hand-written rule (`stale AND visible`) scores **Precision@50 = 0.24**.
 A trained model using the same underlying signals scores **~0.74** — roughly **3x** better. That gap shows there is real, tangled signal in the data that a fixed rule can't capture, but a model can learn.

**Where a rule stays useful:** the rule is still valuable as a transparent baseline and a sanity check — if the model's picks look wildly different from the rule's picks, that's worth investigating. But as the *primary* ranking method, the rule leaves a lot of value on the table.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.